# 🐧 Palmer Penguins — Solutions Notebook
### HPC Summer Workshop 2026

> This notebook contains **full worked solutions** for every exercise in  
> `Palmer_Penguins_Practice.ipynb`.  
> Each solution is wrapped in a collapsible `<details>` block so you can  
> attempt exercises independently before revealing the answer.

---


## Section 1 — Setup & Data Loading

In [ ]:
# Install packages (run once)
pkgs <- c("tidyverse", "caret", "furrr", "future", "broom", "patchwork")
new_pkgs <- pkgs[!pkgs %in% installed.packages()[,"Package"]]
if (length(new_pkgs)) install.packages(new_pkgs, repos = "https://cloud.r-project.org")

library(tidyverse)
library(broom)
library(patchwork)
cat("✅ Packages loaded\n")


### Exercise 1 · Load the data

<details>
<summary>✅ <b>Solution</b> (click to reveal)</summary>

```r
penguins <- read_csv(
  "https://raw.githubusercontent.com/allisonhorst/palmerpenguins/main/inst/extdata/penguins.csv"
)
head(penguins)
```


</details>

In [ ]:
penguins <- read_csv(
  "https://raw.githubusercontent.com/allisonhorst/palmerpenguins/main/inst/extdata/penguins.csv"
)
head(penguins)


### Exercise 2 · Quick look

<details>
<summary>✅ <b>Solution</b> (click to reveal)</summary>

```r
glimpse(penguins)
summary(penguins)
cat("Rows:", nrow(penguins), "  Columns:", ncol(penguins), "\n")
table(penguins$species)
```
The dataset has **344 rows × 8 columns**.  
Species: *Adelie* (152), *Chinstrap* (68), *Gentoo* (124).


</details>

In [ ]:
glimpse(penguins)
summary(penguins)
cat("Rows:", nrow(penguins), " Columns:", ncol(penguins), "\n")
table(penguins$species)


## Section 2 — Exploratory Data Analysis

### Exercise 3 · Missing values

<details>
<summary>✅ <b>Solution</b> (click to reveal)</summary>

```r
colSums(is.na(penguins))
cat("Rows with any NA:", sum(!complete.cases(penguins)), "\n")
```
`bill_length_mm`, `bill_depth_mm`, `flipper_length_mm`, `body_mass_g` each have 2 NAs;  
`sex` has 11 NAs.  Total: **11 rows** have at least one NA.


</details>

In [ ]:
colSums(is.na(penguins))
cat("Rows with any NA:", sum(!complete.cases(penguins)), "\n")


### Exercise 4 · Species × Island frequency table

<details>
<summary>✅ <b>Solution</b> (click to reveal)</summary>

```r
penguins |> count(species, island) |> pivot_wider(names_from = island, values_from = n, values_fill = 0)
```
*Adelie* appears on all three islands; *Chinstrap* only on Dream;  
*Gentoo* only on Biscoe.


</details>

In [ ]:
penguins |>
  count(species, island) |>
  pivot_wider(names_from = island, values_from = n, values_fill = 0)


### Exercise 5 · Body mass by sex

<details>
<summary>✅ <b>Solution</b> (click to reveal)</summary>

```r
penguins |>
  filter(!is.na(sex)) |>
  group_by(species, sex) |>
  summarise(mean_mass = mean(body_mass_g, na.rm = TRUE), .groups = "drop") |>
  arrange(desc(mean_mass))
```
Male Gentoo penguins are the heaviest (≈ 5485 g).


</details>

In [ ]:
penguins |>
  filter(!is.na(sex)) |>
  group_by(species, sex) |>
  summarise(mean_mass = mean(body_mass_g, na.rm = TRUE), .groups = "drop") |>
  arrange(desc(mean_mass))


## Section 3 — Data Wrangling

### Exercise 6 · New columns

<details>
<summary>✅ <b>Solution</b> (click to reveal)</summary>

```r
penguins_aug <- penguins |>
  mutate(
    bill_ratio = bill_length_mm / bill_depth_mm,
    size_class = ifelse(body_mass_g >= 4500, "large", "small")
  )
count(penguins_aug, species, size_class)
```


</details>

In [ ]:
penguins_aug <- penguins |>
  mutate(
    bill_ratio = bill_length_mm / bill_depth_mm,
    size_class = ifelse(body_mass_g >= 4500, "large", "small")
  )
count(penguins_aug, species, size_class)


### Exercise 7 · Pivot longer

<details>
<summary>✅ <b>Solution</b> (click to reveal)</summary>

```r
penguins |>
  select(species, bill_length_mm:body_mass_g) |>
  pivot_longer(-species, names_to = "measurement", values_to = "value") |>
  group_by(species, measurement) |>
  summarise(mean = mean(value, na.rm=TRUE), sd = sd(value, na.rm=TRUE), .groups="drop")
```


</details>

In [ ]:
penguins |>
  select(species, bill_length_mm:body_mass_g) |>
  pivot_longer(-species, names_to = "measurement", values_to = "value") |>
  group_by(species, measurement) |>
  summarise(mean = mean(value, na.rm=TRUE), sd = sd(value, na.rm=TRUE), .groups="drop")


### Exercise 8 · Left join

<details>
<summary>✅ <b>Solution</b> (click to reveal)</summary>

```r
island_info <- tibble(
  island          = c("Torgersen", "Biscoe", "Dream"),
  region          = c("Antarctic Peninsula", "Biscoe Islands", "Dream Island"),
  approx_area_km2 = c(5.4, 64.0, 3.7)
)
penguins |>
  left_join(island_info, by = "island") |>
  group_by(island, region, approx_area_km2) |>
  summarise(avg_mass = mean(body_mass_g, na.rm=TRUE), .groups="drop")
```


</details>

In [ ]:
island_info <- tibble(
  island          = c("Torgersen", "Biscoe", "Dream"),
  region          = c("Antarctic Peninsula", "Biscoe Islands", "Dream Island"),
  approx_area_km2 = c(5.4, 64.0, 3.7)
)
penguins |>
  left_join(island_info, by = "island") |>
  group_by(island, region, approx_area_km2) |>
  summarise(avg_mass = mean(body_mass_g, na.rm=TRUE), .groups="drop")


## Section 4 — Visualisation

### Exercise 9 · Scatter plot

<details>
<summary>✅ <b>Solution</b> (click to reveal)</summary>

```r
p1 <- penguins |>
  filter(complete.cases(bill_length_mm, bill_depth_mm, species)) |>
  ggplot(aes(bill_length_mm, bill_depth_mm, color = species)) +
  geom_point(alpha = 0.7, size = 2) +
  geom_smooth(method = "lm", se = FALSE, linewidth = 1) +
  scale_color_manual(values = c("darkorange","purple","cyan4")) +
  labs(title = "Bill Dimensions by Species",
       x = "Bill Length (mm)", y = "Bill Depth (mm)",
       color = "Species") +
  theme_minimal()
p1
```
The three species form distinct clusters — a classic example of Simpson's Paradox:  
the pooled slope is negative, but each species has a positive trend.


</details>

In [ ]:
p1 <- penguins |>
  filter(complete.cases(bill_length_mm, bill_depth_mm, species)) |>
  ggplot(aes(bill_length_mm, bill_depth_mm, color = species)) +
  geom_point(alpha = 0.7, size = 2) +
  geom_smooth(method = "lm", se = FALSE, linewidth = 1) +
  scale_color_manual(values = c("darkorange","purple","cyan4")) +
  labs(title = "Bill Dimensions by Species",
       x = "Bill Length (mm)", y = "Bill Depth (mm)", color = "Species") +
  theme_minimal()
p1


### Exercise 10 · Box plot

<details>
<summary>✅ <b>Solution</b> (click to reveal)</summary>

```r
p2 <- penguins |> filter(!is.na(sex)) |>
  ggplot(aes(species, body_mass_g, fill = sex)) +
  geom_boxplot(alpha = 0.8) +
  scale_fill_manual(values = c("darkorange","steelblue")) +
  labs(title = "Body Mass by Species and Sex",
       x = "Species", y = "Body Mass (g)") +
  theme_minimal()
p2
```


</details>

In [ ]:
p2 <- penguins |> filter(!is.na(sex)) |>
  ggplot(aes(species, body_mass_g, fill = sex)) +
  geom_boxplot(alpha = 0.8) +
  scale_fill_manual(values = c("darkorange","steelblue")) +
  labs(title = "Body Mass by Species and Sex",
       x = "Species", y = "Body Mass (g)") +
  theme_minimal()
p2


### Exercise 11 · Faceted histograms

<details>
<summary>✅ <b>Solution</b> (click to reveal)</summary>

```r
penguins |> filter(!is.na(flipper_length_mm)) |>
  ggplot(aes(flipper_length_mm, fill = island)) +
  geom_histogram(binwidth = 5, color = "white", alpha = 0.85) +
  facet_wrap(~ species, ncol = 1) +
  labs(title = "Flipper Length Distribution by Species",
       x = "Flipper Length (mm)", y = "Count", fill = "Island") +
  theme_minimal()
```


</details>

In [ ]:
penguins |> filter(!is.na(flipper_length_mm)) |>
  ggplot(aes(flipper_length_mm, fill = island)) +
  geom_histogram(binwidth = 5, color = "white", alpha = 0.85) +
  facet_wrap(~ species, ncol = 1) +
  labs(title = "Flipper Length Distribution by Species",
       x = "Flipper Length (mm)", y = "Count", fill = "Island") +
  theme_minimal()


### Exercise 12 · Patchwork

<details>
<summary>✅ <b>Solution</b> (click to reveal)</summary>

```r
library(patchwork)
p1 + p2 + plot_annotation(
  title = "Palmer Penguins Summary",
  theme = theme(plot.title = element_text(size = 16, face = "bold"))
)
```


</details>

In [ ]:
library(patchwork)
p1 + p2 + plot_annotation(
  title = "Palmer Penguins Summary",
  theme = theme(plot.title = element_text(size = 16, face = "bold"))
)


## Section 5 — Statistical Analysis

### Exercise 13 · Two-sample t-test

<details>
<summary>✅ <b>Solution</b> (click to reveal)</summary>

**H₀:** μ_male = μ_female (no difference in mean body mass for Gentoo penguins)  
**H₁:** μ_male ≠ μ_female

```r
gentoo <- penguins |> filter(species == "Gentoo", !is.na(sex))
t.test(body_mass_g ~ sex, data = gentoo)
```

**Interpretation:** p-value ≪ 0.05 → reject H₀. Male Gentoo penguins are  
significantly heavier than females (≈ 5485 g vs ≈ 4680 g). The 95% CI  
for the difference does not include 0.


</details>

In [ ]:
gentoo <- penguins |> filter(species == "Gentoo", !is.na(sex))
t.test(body_mass_g ~ sex, data = gentoo)


### Exercise 14 · Correlation matrix

<details>
<summary>✅ <b>Solution</b> (click to reveal)</summary>

```r
nums <- penguins |> select(bill_length_mm:body_mass_g) |> drop_na()
cor_mat <- cor(nums)

cor_mat |> as.data.frame() |>
  rownames_to_column("var1") |>
  pivot_longer(-var1, names_to = "var2", values_to = "r") |>
  ggplot(aes(var1, var2, fill = r)) +
  geom_tile(color = "white") +
  geom_text(aes(label = round(r, 2)), size = 4) +
  scale_fill_gradient2(low="blue", high="red", mid="white", midpoint=0, limits=c(-1,1)) +
  labs(title = "Pearson Correlation Matrix", x="", y="") +
  theme_minimal() +
  theme(axis.text.x = element_text(angle=30, hjust=1))
```
Strongest correlation: `flipper_length_mm` ↔ `body_mass_g` (r ≈ 0.87).


</details>

In [ ]:
nums <- penguins |> select(bill_length_mm:body_mass_g) |> drop_na()
cor_mat <- cor(nums)

cor_mat |> as.data.frame() |>
  rownames_to_column("var1") |>
  pivot_longer(-var1, names_to = "var2", values_to = "r") |>
  ggplot(aes(var1, var2, fill = r)) +
  geom_tile(color = "white") +
  geom_text(aes(label = round(r, 2)), size = 4) +
  scale_fill_gradient2(low="blue", high="red", mid="white", midpoint=0, limits=c(-1,1)) +
  labs(title = "Pearson Correlation Matrix", x="", y="") +
  theme_minimal() +
  theme(axis.text.x = element_text(angle=30, hjust=1))


### Exercise 15 · Linear regression

<details>
<summary>✅ <b>Solution</b> (click to reveal)</summary>

```r
model <- lm(body_mass_g ~ flipper_length_mm + species + sex,
            data = drop_na(penguins, body_mass_g, flipper_length_mm, species, sex))
tidy(model)
glance(model)  # R² ~ 0.87
```
`speciesGentoo` has the largest coefficient (≈ +1090 g vs Adelie baseline),  
followed by `sexmale` (≈ +530 g). Adjusted R² ≈ 0.87 — the model explains  
87% of the variance in body mass.


</details>

In [ ]:
model <- lm(body_mass_g ~ flipper_length_mm + species + sex,
            data = drop_na(penguins, body_mass_g, flipper_length_mm, species, sex))
tidy(model)
glance(model)


## Section 6 — Machine Learning

### Exercise 16 · Prepare ML data

<details>
<summary>✅ <b>Solution</b> (click to reveal)</summary>

```r
set.seed(42)
penguins_ml <- penguins |>
  select(species, bill_length_mm:body_mass_g) |>
  drop_na() |>
  mutate(species = factor(species))

idx   <- sample(nrow(penguins_ml), 0.8 * nrow(penguins_ml))
train <- penguins_ml[idx, ]
test  <- penguins_ml[-idx, ]
table(train$species)
```


</details>

In [ ]:
set.seed(42)
penguins_ml <- penguins |>
  select(species, bill_length_mm:body_mass_g) |>
  drop_na() |>
  mutate(species = factor(species))

idx   <- sample(nrow(penguins_ml), 0.8 * nrow(penguins_ml))
train <- penguins_ml[idx, ]
test  <- penguins_ml[-idx, ]
cat("Training set size:", nrow(train), "\n")
table(train$species)


### Exercise 17 · Train k-NN

<details>
<summary>✅ <b>Solution</b> (click to reveal)</summary>

```r
library(caret)
ctrl      <- trainControl(method = "cv", number = 5)
kgrid     <- expand.grid(k = c(3, 5, 7, 9, 11))
knn_model <- train(species ~ ., data = train,
                   method     = "knn",
                   trControl  = ctrl,
                   tuneGrid   = kgrid,
                   preProcess = c("center", "scale"))
print(knn_model)
plot(knn_model)
```
The best `k` is usually 5 or 7, with cross-validation accuracy > 98%.


</details>

In [ ]:
library(caret)
ctrl      <- trainControl(method = "cv", number = 5)
kgrid     <- expand.grid(k = c(3, 5, 7, 9, 11))
knn_model <- train(species ~ ., data = train,
                   method     = "knn",
                   trControl  = ctrl,
                   tuneGrid   = kgrid,
                   preProcess = c("center", "scale"))
print(knn_model)


### Exercise 18 · Evaluate on test set

<details>
<summary>✅ <b>Solution</b> (click to reveal)</summary>

```r
preds <- predict(knn_model, newdata = test)
confusionMatrix(preds, test$species)
```
Overall accuracy is typically **~98%**. Chinstrap is occasionally confused  
with Adelie because their bill lengths overlap more than other pairs.


</details>

In [ ]:
preds <- predict(knn_model, newdata = test)
confusionMatrix(preds, test$species)


## Section 7 — Parallel Bootstrap

<details>
<summary>✅ <b>Solution</b> (click to reveal)</summary>

```r
library(furrr); library(future)

gentoo_flip <- penguins |>
  filter(species == "Gentoo", !is.na(flipper_length_mm)) |>
  pull(flipper_length_mm)

set.seed(99)
boot_fn <- function(i) mean(sample(gentoo_flip, replace = TRUE))

# Sequential
t_seq <- system.time(boot_seq <- sapply(1:1000, boot_fn))

# Parallel
plan(multisession, workers = 4)
t_par <- system.time(boot_par <- future_map_dbl(1:1000, boot_fn))
plan(sequential)

cat("Sequential time:", t_seq["elapsed"], "s\n")
cat("Parallel time:  ", t_par["elapsed"], "s\n")
cat("95% CI: [", quantile(boot_par, 0.025), ",", quantile(boot_par, 0.975), "]\n")
```
**Note:** For only 1000 resamples the overhead of spawning workers may  
dominate. Parallelism pays off more clearly with 10,000+ resamples or  
heavier per-resample computation — typical in HPC workflows.


</details>

In [ ]:
library(furrr)
library(future)

gentoo_flip <- penguins |>
  filter(species == "Gentoo", !is.na(flipper_length_mm)) |>
  pull(flipper_length_mm)

set.seed(99)
boot_fn <- function(i) mean(sample(gentoo_flip, replace = TRUE))

# Sequential
t_seq <- system.time(boot_seq <- sapply(1:1000, boot_fn))

# Parallel
plan(multisession, workers = 4)
t_par <- system.time(boot_par <- future_map_dbl(1:1000, boot_fn))
plan(sequential)

cat("Sequential time:", t_seq["elapsed"], "s\n")
cat("Parallel time:  ", t_par["elapsed"], "s\n")
cat("95% CI: [", round(quantile(boot_par, 0.025),2), ",",
    round(quantile(boot_par, 0.975),2), "] mm\n")


## Section 8 — AI Collaboration

### Exercise 20 · Improved prompts

<details>
<summary>✅ <b>Solution</b> (click to reveal)</summary>

**A — Clean the data:**  
> *"I have the palmerpenguins dataset loaded as a tibble in R. It has 344 rows  
> and columns: species, island, bill_length_mm, bill_depth_mm, flipper_length_mm,  
> body_mass_g, sex, year. Please write tidyverse R code to: (1) report how many  
> NAs are in each column, (2) drop rows where sex is NA, and (3) confirm the  
> cleaned dataset has no remaining NAs in any column."*

**B — Make a nice chart:**  
> *"Using ggplot2 in R, create a scatter plot of bill_length_mm (x-axis) vs  
> body_mass_g (y-axis) from the penguins tibble. Color points by species using  
> the palette c('darkorange','purple','cyan4'). Add a linear regression line  
> per species (geom_smooth, no confidence band). Use theme_minimal(), add axis  
> labels, and a descriptive title."*

**C — Do statistics on penguins:**  
> *"In R using the penguins dataset, run a Welch two-sample t-test comparing  
> mean flipper_length_mm between Adelie and Chinstrap species (drop NAs first).  
> Show the test output and interpret: state H₀, the p-value, and whether to  
> reject at α = 0.05."*

**Key elements of an effective data-science prompt:**  
dataset name + structure, target language/package, exact column names,  
desired output format, statistical method name, interpretation expectations.


</details>

In [ ]:
# Reflection answers written above in the solution block.
# For Exercise 20 there is no runnable code — the exercise is about prompt quality.
cat("See the solution block above for improved prompts and discussion.\n")


### Exercise 21 · Bug fixes

<details>
<summary>✅ <b>Solution</b> (click to reveal)</summary>

Bugs and fixes:
1. `library(Tidyverse)` → `library(tidyverse)` (R is case-sensitive)
2. `!is.na(Sex)` → `!is.na(sex)` (column is lowercase)
3. `average(...)` → `mean(..., na.rm = TRUE)` (no `average()` in base R)
4. `avg_flippr` → `avg_flipper` (typo in `arrange()`)


</details>

In [ ]:
# FIXED CODE
library(tidyverse)   # Fix 1: lowercase

penguins |>
  filter(!is.na(sex)) |>                              # Fix 2: lowercase sex
  group_by(species, sex) |>
  summarise(
    avg_flipper = mean(flipper_length_mm, na.rm=TRUE), # Fix 3: mean() + na.rm
    n = n(),
    .groups = "drop"
  ) |>
  arrange(desc(avg_flipper))                           # Fix 4: correct name


### Exercise 22 · AI code review

<details>
<summary>✅ <b>Solution</b> (click to reveal)</summary>

Original issues:
1. `bill_length` → should be `bill_length_mm`
2. `body_mass` → should be `body_mass_g`
3. `facet_wrap(species)` → needs `facet_wrap(~ species)`
4. No NA handling
5. No axis labels or title; no color for clarity

```r
# Improved version
penguins |>
  filter(!is.na(bill_length_mm), !is.na(body_mass_g)) |>   # Fix 4: drop NAs
  ggplot(aes(bill_length_mm, body_mass_g, color = species)) + # Fix 1,2 + color
  geom_point(alpha = 0.7) +
  facet_wrap(~ species) +                                    # Fix 3: tilde
  scale_color_manual(values = c("darkorange","purple","cyan4")) +
  labs(title = "Bill Length vs Body Mass by Species",        # Fix 5: labels
       x = "Bill Length (mm)", y = "Body Mass (g)") +
  theme_minimal() +
  theme(legend.position = "none")  # color already shown by facet title
```


</details>

In [ ]:
penguins |>
  filter(!is.na(bill_length_mm), !is.na(body_mass_g)) |>
  ggplot(aes(bill_length_mm, body_mass_g, color = species)) +
  geom_point(alpha = 0.7) +
  facet_wrap(~ species) +
  scale_color_manual(values = c("darkorange","purple","cyan4")) +
  labs(title = "Bill Length vs Body Mass by Species",
       x = "Bill Length (mm)", y = "Body Mass (g)") +
  theme_minimal() +
  theme(legend.position = "none")


---
## ✅ End of Solutions

All exercises covered. Remember:  
- Practice notebook: `Palmer_Penguins_Practice.ipynb`  
- Dataset: Palmer Penguins (CC-0 license) — Gorman, Williams & Fraser (2014)
